In [ ]:
!pip install arch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

In [ ]:
df = pd.read_excel("/content/USA CP2.xlsx")
print(df.head())


# графики рядов

In [ ]:
df.set_index('Year')[['GDP', 'CO2', 'Credit']].plot(subplots=True, figsize=(10, 6), title='Временные ряды')
plt.tight_layout()
plt.show()

**Ряд GDP** демонстрирует устойчивый восходящий тренд, что указывает на его нестабильность и возможную нестационарность.

**Ряд CO₂** имеет волнообразную структуру, без ярко выраженного тренда, но возможна слабая нисходящая тенденция. Это требует подтверждения тестами.

**Ряд Credit** также демонстрирует долгосрочный тренд роста, что указывает на его нестационарность.

# ADF-тест

In [ ]:
def adf_test(series):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"ADF Test:")
    print(f"  Статистика = {result[0]:.4f}")
    print(f"  p-value = {result[1]:.4f}")
    print(f"  Критические значения: {result[4]}")
    print("  Вывод: Стационарен" if result[1] < 0.05 else "  Вывод: НЕстационарен")

# PP-тест

In [ ]:
def pp_test(series):
    pp = PhillipsPerron(series.dropna())
    print("PP Test:")
    print(f"  Статистика = {pp.stat:.4f}")
    print(f"  p-value = {pp.pvalue:.4f}")
    print("  Вывод: Стационарен" if pp.pvalue < 0.05 else "  Вывод: НЕстационарен")

# KPSS-тест

In [ ]:
def kpss_test(series):
    result = kpss(series.dropna(), regression='c', nlags="auto")
    print(f"KPSS Test:")
    print(f"  Статистика = {result[0]:.4f}")
    print(f"  p-value = {result[1]:.4f}")
    print(f"  Критические значения: {result[3]}")
    print("  Вывод: Стационарен" if result[1] > 0.05 else "  Вывод: НЕстационарен")

# Проверка исходных рядов

In [ ]:
print("=== GDP ===")
adf_test(df['GDP'])
pp_test(df['GDP'])
kpss_test(df['GDP'])

print("\n=== CO2 ===")
adf_test(df['CO2'])
pp_test(df['CO2'])
kpss_test(df['CO2'])

print("\n=== Credit ===")
adf_test(df['Credit'])
pp_test(df['Credit'])
kpss_test(df['Credit'])

# Проверка разностей


In [ ]:
df['GDP_diff'] = df['GDP'].diff()
df['CO2_diff'] = df['CO2'].diff()
df['Credit_diff'] = df['Credit'].diff()

In [ ]:
print("\n=== ΔGDP ===")
adf_test(df['GDP_diff'])
pp_test(df['GDP_diff'])
kpss_test(df['GDP_diff'])

print("\n=== ΔCO2 ===")
adf_test(df['CO2_diff'])
pp_test(df['CO2_diff'])
kpss_test(df['CO2_diff'])

print("\n=== ΔCredit ===")
adf_test(df['Credit_diff'])
pp_test(df['Credit_diff'])
kpss_test(df['Credit_diff'])

## ИТОГ

| Показатель | Тест 1 (ADF)                | Тест 2 (PP)                 | Тест 3 (KPSS)               | Вывод о стационарности / порядке интегрируемости |
| ---------- | --------------------------- | --------------------------- | --------------------------- | ------------------------------------------------ |
| GDP        | p = 0.897, H₀: не отклонена | p = 0.880, H₀: не отклонена | p = 0.010, H₀: отклонена    | НЕстационарен, I(1)                              |
| ΔGDP       | p = 0.0002, H₀: отклонена   | p = 0.0005, H₀: отклонена   | p = 0.100, H₀: не отклонена |  Стационарен, I(0)                              |
| CO₂        | p = 0.850, H₀: не отклонена | p = 0.898, H₀: не отклонена | p = 0.025, H₀: отклонена    | НЕстационарен, I(1)                              |
| ΔCO₂       | p = 0.033, H₀: отклонена    | p = 0.0000, H₀: отклонена   | p = 0.100, H₀: не отклонена |  Стационарен, I(0)                              |
| Credit     | p = 0.473, H₀: не отклонена | p = 0.884, H₀: не отклонена | p = 0.010, H₀: отклонена    | НЕстационарен, I(1)                              |
| ΔCredit    | p = 0.299, H₀: не отклонена | p = 0.0000, H₀: отклонена   | p = 0.100, H₀: не отклонена |  Скорее стационарен, I(0)                      |


# **************заметка
| Тест     | H₀ (нулевая гипотеза)                         | Что значит «отклонена» |
| -------- | --------------------------------------------- | ---------------------- |
| **ADF**  | Ряд **нестационарен** (есть единичный корень) | Ряд **стационарен**    |
| **PP**   | Ряд **нестационарен**                         | Ряд **стационарен**    |
| **KPSS** | Ряд **стационарен**                           | Ряд **нестационарен**  |


# Анализ стационарности временных рядов
Для всех трёх переменных (GDP, CO2, Credit) была проверена стационарность с помощью трёх популярных тестов: ADF (тест Дики-Фуллера), PP (тест Филипса-Перрона) и KPSS.

Результаты анализа исходных рядов показали, что все три временных ряда являются нестационарными: значения p-value ADF и PP значительно превышают 0.05, а тест KPSS отвергает гипотезу о стационарности. Это означает наличие трендов и требует дальнейшего преобразования.

После преобразования рядов к первой разности:

*   GDP и CO₂ стали однозначно стационарными — все три теста это подтвердили.
*   Credit показал смешанные результаты: ADF не отверг гипотезу о нестационарности, однако тесты PP и KPSS подтвердили стационарность. Это позволяет предположить, что ряд можно считать стационарным с оговорками.

Итог: все ряды имеют порядок интегрируемости I(1), что позволяет переходить к коинтеграционному анализу и использованию моделей VAR, ECM или VECM.

# Исследовательский вопрос:

Сохраняется ли в США выявленная для Пакистана коинтеграция между уровнем экономического развития, объёмом кредитования и выбросами CO₂?

Выбор модели:
Так как все три переменные являются интегрированными порядка I(1), и в исследовательском вопросе предполагается наличие долгосрочной связи между ними, было решено использовать тест Джохансена для проверки наличия коинтеграции. В случае её подтверждения в анализе будет использоваться модель коррекции ошибок (VECM), которая позволяет одновременно моделировать долгосрочное равновесие и краткосрочные колебания между рядами.
Если же коинтеграция не будет выявлена, будет использована модель VAR по разностным рядам.

In [ ]:
!pip install statsmodels

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

df_johansen = df[['GDP', 'CO2', 'Credit']].dropna()
johansen_result = coint_johansen(df_johansen, det_order=0, k_ar_diff=1)

print("Johansen Trace Test:")
print("Статистика:", johansen_result.lr1)
print("Критические значения (90%, 95%, 99%):")
print(johansen_result.cvt)

| Кол-во коинтеграционных векторов (rank) | Trace statistic | 95% крит. значение | Вывод                                  |
| --------------------------------------- | --------------- | ------------------ | -------------------------------------- |
| **0** (нет связи)                       | **20.39**       | **29.80**          |  H₀ не отвергается — коинтеграции нет |
| **≤ 1**                                 | **5.45**        | **15.49**          |  H₀ не отвергается                    |
| **≤ 2**                                 | **0.04**        | **3.84**           |  H₀ не отвергается                    |


Результаты теста Джохансена на коинтеграцию показали отсутствие коинтеграционных связей между переменными GDP, CO₂ и Credit. Значения статистик теста оказались ниже критических значений на всех уровнях значимости (10%, 5%, 1%). Это указывает на то, что ряды не демонстрируют устойчивой долгосрочной взаимосвязи.

Следовательно, для моделирования взаимосвязей между переменными используется модель VAR, построенная на первых разностях временных рядов.

In [ ]:
from statsmodels.tsa.api import VAR
df_var = df[['GDP_diff', 'CO2_diff', 'Credit_diff']].dropna()

In [ ]:
model = VAR(df_var)
lag_results = model.select_order(maxlags=3)
print(lag_results.summary())

Минимальные значения всех критериев при 1 лаге. Значит, оптимальный лаг -- 1

In [ ]:
model_fitted = model.fit(1)
print(model_fitted.summary())

проверяю устойчевость модели

In [ ]:
print(model_fitted.is_stable())

In [ ]:
irf = model_fitted.irf(10)
irf.plot(orth=False)

In [ ]:
fevd = model_fitted.fevd(10)
fevd.plot()

Моделирование взаимосвязей во временных рядах

В работе построена модель векторной авторегрессии (VAR) на первых разностях переменных GDP, CO₂ и Credit, так как исходные ряды оказались нестационарными и некоинтегрированными (по результатам теста Джохансена).

Оптимальное количество лагов (1) определено на основе информационных критериев (AIC, BIC, FPE). Модель оказалась устойчивой (is_stable() = True).
Результаты VAR-модели:

*   Кредитование (Credit_diff) оказывает значимое положительное влияние на экономический рост (GDP_diff, p < 0.001).
*    Кредитование также оказывает значимое положительное влияние на выбросы CO₂ (CO2_diff, p < 0.01).
*   В свою очередь, рост ВВП стимулирует рост кредитования (p < 0.01), что отражает внутреннюю циклическую связь между экономикой и финансовым сектором.

Импульсные отклики (IRF):

Графики импульсных откликов показали, что:

*   Шок в кредитовании вызывает заметную положительную реакцию как ВВП, так и выбросов CO₂ — особенно в первые 2–3 года.

*   Шок в GDP приводит к росту кредитования и незначительно влияет на выбросы CO₂.

Это подтверждает гипотезу Abbasi & Riaz (2016), что финансовое развитие способствует увеличению выбросов CO₂, но теперь это показано на примере США.
Декомпозиция дисперсии (FEVD):

*   Вариация ВВП объясняется в основном им самим (~70%), но вклад кредитования устойчиво значим.

*   Выбросы CO₂ объясняются как ВВП, так и кредитами, но влияние кредитов заметно.

*   Кредиты в основном объясняются самими собой и ростом ВВП.

**Итог:** результаты модели VAR показывают наличие краткосрочных связей между переменными и подтверждают применимость гипотезы экологического воздействия финансового сектора и экономического роста в контексте США.